## Libraries Imports

In [1]:
# %pip install -r requirements.txt

In [2]:
import plotly.express as px
import pandas as pd

## Dataset Import

In [3]:
df_titanic = pd.read_csv("data/titanic.csv")

## Data Preprocessing

In [4]:
df_titanic.head()

,name,gender,age,class,embarked,country,ticketno,fare,sibsp,parch,survived
0,"Abbing, Mr. Anthony",male,42.0,3rd,S,United States,5547.0,7.11,0.0,0.0,no
1,"Abbott, Mr. Eugene Joseph",male,13.0,3rd,S,United States,2673.0,20.05,0.0,2.0,no
2,"Abbott, Mr. Rossmore Edward",male,16.0,3rd,S,United States,2673.0,20.05,1.0,1.0,no
3,"Abbott, Mrs. Rhoda Mary 'Rosa'",female,39.0,3rd,S,England,2673.0,20.05,1.0,1.0,yes
4,"Abelseth, Miss. Karen Marie",female,16.0,3rd,S,Norway,348125.0,7.13,0.0,0.0,yes


In [5]:
# Keeping only passengers
PORTS_MAPPING = {'S': 'Southampton', 'C': 'Cherbourg', 'Q': 'Queenstown'}
PASSENGERS_CLASS = ['1st', '2nd', '3rd']
CLASS_MAPPING = {'1st': '1ère', '2nd': '2ème', '3rd': '3ème'}

df_passengers = df_titanic[df_titanic['embarked'] != 'B']
df_passengers.loc[:, 'embarked'] = df_passengers['embarked'].map(PORTS_MAPPING)

df_passengers = df_passengers[df_passengers['class'].isin(PASSENGERS_CLASS)]
df_passengers.loc[:, 'class'] = df_passengers['class'].map(CLASS_MAPPING)

In [6]:
SURVIVED_MAPPING = {'yes': 'oui', 'no': 'non'}
df_passengers.loc[:, 'survived'] = df_passengers['survived'].map(SURVIVED_MAPPING)

df_passengers.head()

,name,gender,age,class,embarked,country,ticketno,fare,sibsp,parch,survived
0,"Abbing, Mr. Anthony",male,42.0,3ème,Southampton,United States,5547.0,7.11,0.0,0.0,non
1,"Abbott, Mr. Eugene Joseph",male,13.0,3ème,Southampton,United States,2673.0,20.05,0.0,2.0,non
2,"Abbott, Mr. Rossmore Edward",male,16.0,3ème,Southampton,United States,2673.0,20.05,1.0,1.0,non
3,"Abbott, Mrs. Rhoda Mary 'Rosa'",female,39.0,3ème,Southampton,England,2673.0,20.05,1.0,1.0,oui
4,"Abelseth, Miss. Karen Marie",female,16.0,3ème,Southampton,Norway,348125.0,7.13,0.0,0.0,oui


In [ ]:
# Get number of survivors and non-survivors per class and port
df_survivors = df_passengers.groupby(['embarked', 'class', 'survived']).size().unstack(fill_value=0)

df_survivors['total_survivors'] = df_survivors.get('oui', 0)
df_survivors['total_non_survivors'] = df_survivors.get('non', 0)

df_survivors = df_survivors.reset_index()

# Merge in main df
df_passengers = df_passengers.merge(
    df_survivors[['embarked', 'class', 'total_survivors', 'total_non_survivors']],
    on=['embarked', 'class'],
    how='left'
)

## Mockup Plots

In [30]:
SURVIVOR_COLOR = 'teal'
NON_SURVIVOR_COLOR = 'tomato'

ports = df_passengers['embarked'].unique()

for port in ports:
    df_port = df_passengers[df_passengers['embarked'] == port]

    fig = px.box(
        df_port,
        x='class',
        y='fare',
        color='survived',
        points='all',
        hover_data=['embarked'],
        labels={
            'class': 'Classe',
            'fare': 'Prix du billet',
            'survived': 'Survie',
            'embarked': 'Port d\'embarquement',
        },
        facet_col='embarked',
        color_discrete_map={'non': NON_SURVIVOR_COLOR, 'oui': SURVIVOR_COLOR}
    )

    # Light Background Theme
    LIGHT_GRAY_BG = 'rgba(245, 245, 245, 1)'
    WHITE_BG = 'rgba(255, 255, 255, 1)'
    GRAY_GRID = 'rgba(200, 200, 200, 0.5)'

    fig.update_layout(
        plot_bgcolor=LIGHT_GRAY_BG,
        paper_bgcolor=WHITE_BG,
        xaxis=dict(showgrid=True, gridcolor=GRAY_GRID),
        yaxis=dict(showgrid=True, gridcolor=GRAY_GRID),
    )

    # Display number of survivors and non-survivors directly
    FARE_MAX = df_port['fare'].max()
    FARE_MIN = df_port['fare'].min()
    FARE_RANGE = FARE_MAX - FARE_MIN
    OFFSET_Y_TOP_LABEL = FARE_RANGE * 0.4
    OFFSET_Y_BOTTOM_LABEL = FARE_RANGE * 0.3

    for i, row in df_survivors[df_survivors['embarked'] == port].iterrows():
        fig.add_annotation(
            x=row['class'],
            y=df_port[df_port['class'] == row['class']]['fare'].max() + OFFSET_Y_TOP_LABEL,
            text=f"<span style='color:{SURVIVOR_COLOR}'>Survivants: {row['total_survivors']}</span>",
            showarrow=False,
            font=dict(size=12),
            align='center'
        )
        fig.add_annotation(
            x=row['class'],
            y=df_port[df_port['class'] == row['class']]['fare'].max() + OFFSET_Y_BOTTOM_LABEL,
            text=f"<span style='color:{NON_SURVIVOR_COLOR}'>Naufragés: {row['total_non_survivors']}</span>",
            showarrow=False,
            font=dict(size=12),
            align='center'
        )

    fig.show()